# Evaluación 1: SQL
En este evaluación deberán:
1. Completar cada una de las consultas pedidas a continuación
2. Escribir una expresión equivalente en algebra relacional para las consultas 1 a 6.

**[IMPORTANTE]**

El algebra relacional se debe entregar en el mismo buzón que este notebook pero en un **documento aparte** en formato **pdf** escrito en el editor de texto que prefieran (Latex, Typst, Word, a mano, etc.).

El esquema que utilizarán es el siguiente:
```
actores (
    id SERIAL PRIMARY KEY,
    anombre VARCHAR(100) NOT NULL
);
peliculas (
    id SERIAL PRIMARY KEY,
    pnombre VARCHAR(100) NOT NULL,
    año INT NOT NULL,
    categoria VARCHAR(50) NOT NULL,
    calificacion NUMERIC NOT NULL,
    pdirector VARCHAR(100) NOT NULL
);
actuo_en (
    id_actor INT NOT NULL,
    id_pelicula INT NOT NULL,
    FOREIGN KEY (id_actor) REFERENCES actores(id),
    FOREIGN KEY (id_pelicula) REFERENCES peliculas(id)
);
```

## Setup inicial
Creamos y configuramos las tablas de la base de datos

In [1]:
import subprocess, sys, pathlib
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
import gdown
if not pathlib.Path("db_ev1.sql").exists():
    gdown.download("https://drive.google.com/uc?id=1_OaysJsdTr8c2ToecOWXs9KyIBcK1H5f", "db_ev1.sql", quiet=False)
print("db_ev1.sql listo")

db_ev1.sql listo


In [2]:
import subprocess, sys, sqlite3, pathlib

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipython-sql", "sqlalchemy"])

db_path = pathlib.Path("ev1.db")
if db_path.exists():
    db_path.unlink()

conn = sqlite3.connect(str(db_path))
cur = conn.cursor()

cur.execute("CREATE TABLE actores (id INTEGER PRIMARY KEY, anombre TEXT NOT NULL)")
cur.execute('CREATE TABLE peliculas (id INTEGER PRIMARY KEY, pnombre TEXT NOT NULL, "año" INTEGER NOT NULL, categoria TEXT NOT NULL, calificacion REAL NOT NULL, pdirector TEXT NOT NULL)')
cur.execute("CREATE TABLE actuo_en (id_actor INTEGER NOT NULL, id_pelicula INTEGER NOT NULL, FOREIGN KEY (id_actor) REFERENCES actores(id), FOREIGN KEY (id_pelicula) REFERENCES peliculas(id))")

lines = pathlib.Path("db_ev1.sql").read_text(encoding="utf-8").splitlines()
i, n = 0, len(lines)
while i < n:
    line = lines[i]
    if line.startswith("COPY public.actores "):
        i += 1
        while i < n and lines[i] != '\\.':
            parts = lines[i].split("\t")
            cur.execute("INSERT INTO actores VALUES (?,?)", (int(parts[0]), parts[1]))
            i += 1
    elif line.startswith("COPY public.peliculas "):
        i += 1
        while i < n and lines[i] != '\\.':
            parts = lines[i].split("\t")
            cur.execute("INSERT INTO peliculas VALUES (?,?,?,?,?,?)", (int(parts[0]), parts[1], int(parts[2]), parts[3], float(parts[4]), parts[5]))
            i += 1
    elif line.startswith("COPY public.actuo_en "):
        i += 1
        while i < n and lines[i] != '\\.':
            parts = lines[i].split("\t")
            cur.execute("INSERT INTO actuo_en VALUES (?,?)", (int(parts[0]), int(parts[1])))
            i += 1
    i += 1

conn.commit()
for t in ["actores","peliculas","actuo_en"]:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"{t}: {cur.fetchone()[0]} filas")
conn.close()

%load_ext sql
%config SqlMagic.feedback = False
%config SqlMagic.autopandas = True
%config SqlMagic.style = "_DEPRECATED_DEFAULT"
%sql sqlite:///ev1.db

actores: 8 filas
peliculas: 8 filas
actuo_en: 10 filas


Ejecuta la celda de abajo para confirmar que se cargaron bien las tablas.

In [3]:
%%sql
SELECT * FROM peliculas;

 * sqlite:///ev1.db


,id,pnombre,año,categoria,calificacion,pdirector
0,1,Interstellar,2014,SciFi,8.6,C. Nolan
1,2,The Revenant,2015,Drama,8.1,A. Iñárritu
2,3,Harry Potter,2011,Fantasy,8.1,D. Yates
3,4,The Theory of Everything,2014,Biography,7.7,J. Marsh
4,5,Inception,2010,Adventure,8.8,C. Nolan
5,6,Forrest Gump,1994,Drama,8.8,R. Zemeckis
6,7,Lost in Translation,2003,Drama,7.8,S. Coppola
7,8,Fight Club,1999,Drama,8.8,D. Fincher


## Consultas
A continuación se detallan las consultas que debes completar

### Consulta 1: Encuentre los nombres de todos los actores

Recuerda que tambien debes crear una expresión de algebra relacional equivalente a esta consulta.

In [4]:
%%sql
SELECT anombre FROM actores;

 * sqlite:///ev1.db


,anombre
0,Leonardo DiCaprio
1,Matthew McConaughey
2,Daniel Radcliffe
3,Jessica Chastain
4,Tom Hanks
5,Scarlett Johansson
6,Meryl Streep
7,Brad Pitt


### Consulta 2: Encuentre los nombres de todas las películas que fueron dirigidas por 'C. Nolan'.

Recuerda que tambien debes crear una expresión de algebra relacional equivalente a esta consulta.

In [5]:
%%sql
SELECT pnombre
FROM peliculas
WHERE pdirector = 'C. Nolan';

 * sqlite:///ev1.db


,pnombre
0,Interstellar
1,Inception


### Consulta 3: Encuentre los nombres de todas las películas que se estrenaron después del año 2010.

Recuerda que tambien debes crear una expresión de algebra relacional equivalente a esta consulta.

In [6]:
%%sql
SELECT pnombre
FROM peliculas
WHERE "año" > 2010;

 * sqlite:///ev1.db


,pnombre
0,Interstellar
1,The Revenant
2,Harry Potter
3,The Theory of Everything


### Consulta 4: Encuentre los nombres de todas las películas de la categoría 'SciFi' que tienen una calificación mayor a 8.5.

Recuerda que tambien debes crear una expresión de algebra relacional equivalente a esta consulta.

In [7]:
%%sql
SELECT pnombre
FROM peliculas
WHERE categoria = 'SciFi' AND calificacion > 8.5;

 * sqlite:///ev1.db


,pnombre
0,Interstellar


### Consulta 5: Encuentre los identificadores (id) de todos los actores que actuaron en la película con nombre 'Interstellar'.

Recuerda que tambien debes crear una expresión de algebra relacional equivalente a esta consulta.

In [8]:
%%sql
SELECT ae.id_actor
FROM actuo_en ae
JOIN peliculas p ON ae.id_pelicula = p.id
WHERE p.pnombre = 'Interstellar';

 * sqlite:///ev1.db


,id_actor
0,2
1,4


### Consulta 6: Encuentre los nombres de todos los actores que actuaron en alguna película dirigida por 'C. Nolan'.

Recuerda que tambien debes crear una expresión de algebra relacional equivalente a esta consulta.

In [9]:
%%sql
SELECT DISTINCT a.anombre
FROM actores a
JOIN actuo_en ae ON a.id = ae.id_actor
JOIN peliculas p ON ae.id_pelicula = p.id
WHERE p.pdirector = 'C. Nolan';

 * sqlite:///ev1.db


,anombre
0,Matthew McConaughey
1,Jessica Chastain
2,Leonardo DiCaprio
3,Scarlett Johansson


### Consulta 7: Encuentre los actores que no actuaron en ninguna película dirigida por 'C. Nolan'.



In [10]:
%%sql
SELECT a.anombre
FROM actores a
WHERE a.id NOT IN (
    SELECT ae.id_actor
    FROM actuo_en ae
    JOIN peliculas p ON ae.id_pelicula = p.id
    WHERE p.pdirector = 'C. Nolan'
);

 * sqlite:///ev1.db


,anombre
0,Daniel Radcliffe
1,Tom Hanks
2,Meryl Streep
3,Brad Pitt


### Consulta 8: Encuentra los actores que solo actuaron en películas de categoría 'SciFi' o 'Adventure'.

In [11]:
%%sql
SELECT a.anombre
FROM actores a
JOIN actuo_en ae ON a.id = ae.id_actor
WHERE NOT EXISTS (
    SELECT 1
    FROM actuo_en ae2
    JOIN peliculas p ON ae2.id_pelicula = p.id
    WHERE ae2.id_actor = a.id
      AND p.categoria NOT IN ('SciFi', 'Adventure')
)
GROUP BY a.id, a.anombre;

 * sqlite:///ev1.db


,anombre
0,Matthew McConaughey
1,Jessica Chastain


### Consulta 9: Actores que han trabajado con más de un director.

In [12]:
%%sql
SELECT a.anombre
FROM actores a
JOIN actuo_en ae ON a.id = ae.id_actor
JOIN peliculas p ON ae.id_pelicula = p.id
GROUP BY a.id, a.anombre
HAVING COUNT(DISTINCT p.pdirector) > 1;

 * sqlite:///ev1.db


,anombre
0,Leonardo DiCaprio
1,Scarlett Johansson


### Consulta 10: Parejas de actores que trabajaron juntos en al menos una película.

In [ ]:
%%sql
SELECT DISTINCT a1.anombre AS actor1, a2.anombre AS actor2
FROM actuo_en ae1
JOIN actuo_en ae2 ON ae1.id_pelicula = ae2.id_pelicula AND ae1.id_actor < ae2.id_actor
JOIN actores a1 ON ae1.id_actor = a1.id
JOIN actores a2 ON ae2.id_actor = a2.id;

 * sqlite:///ev1.db


,actor1,actor2
0,Matthew McConaughey,Jessica Chastain
1,Leonardo DiCaprio,Scarlett Johansson
2,Meryl Streep,Brad Pitt


: 